# Shor Algorithm — Teaching Notebook

Shor’s factoring algorithm reduces integer factorization to **order finding**. The quantum part estimates the period of modular exponentiation using phase estimation / Fourier interference; the classical part converts that period into factors.

This notebook is designed to be **self-explanatory for classroom teaching**. It moves from the problem statement and mathematics to quantum circuits, Qiskit implementation, interpretation, limitations, and exercises.

### Learning objectives
- understand the reduction from factoring to order finding
- derive modular periodicity
- connect order finding to QPE and inverse QFT
- implement a pedagogical N=15 example
- perform continued-fraction post-processing
- distinguish the quantum and classical parts of Shor

## 0. Installation
Run this only if Qiskit is not already installed.

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib numpy

## 1. Imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, pi

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title='Measurement counts'):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8,4))
    plt.bar(keys, vals)
    plt.xlabel('bit string')
    plt.ylabel('counts')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Part I — Factoring as order finding
Given a composite integer $N$, choose $a$ with

$$1<a<N,\qquad \gcd(a,N)=1.$$

The **order** $r$ of $a$ modulo $N$ is the smallest positive integer satisfying

$$a^r\equiv1\pmod N.$$

If $r$ is even and $a^{r/2}\not\equiv-1\pmod N$, then

$$a^r-1=(a^{r/2}-1)(a^{r/2}+1)$$

is divisible by $N$, so nontrivial factors can often be obtained from

$$\gcd(a^{r/2}-1,N),\qquad \gcd(a^{r/2}+1,N).$$

## 2. Classical preprocessing example: $N=15$
Choose $a=2$:

$$2^1=2,\;2^2=4,\;2^3=8,\;2^4=16\equiv1\pmod{15}.$$

Thus $r=4$. Then

$$\gcd(2^2-1,15)=3,\qquad \gcd(2^2+1,15)=5.$$

In [ ]:
N=15; a=2
vals=[]
x=1
for k in range(8):
    vals.append(x)
    x=(x*a)%N
print(vals)

# Part II — Where the quantum computer is used
The difficult step is finding $r$ efficiently for large $N$. Shor uses a quantum circuit to detect the periodic structure of

$$f(x)=a^x\bmod N.$$

A conceptually simple period-finding state is

$$\frac1{\sqrt Q}\sum_{x=0}^{Q-1}|x\rangle|a^x\bmod N\rangle.$$

The first register contains a periodic pattern correlated with the second register. A Fourier transform turns that periodicity into sharp frequency peaks.

## 3. Periodicity to frequency
For an ideal period $r$, QFT measurement values concentrate near

$$\frac{kQ}{r},\qquad k=0,1,\dots,r-1.$$

Dividing a measured value $m$ by $Q$ gives approximately

$$\frac mQ\approx\frac kr.$$

A continued-fraction expansion can recover $r$.

# Part III — QPE interpretation
Modular multiplication by $a$ defines a unitary

$$U_a|y\rangle=|ay\bmod N\rangle$$

on the coprime residue subspace. Its eigenvalues are $e^{2\pi i k/r}$. Quantum Phase Estimation estimates those eigenphases. The inverse QFT acts as the **phase decoder**.

## 4. Manual inverse QFT helper

In [ ]:
def inverse_qft(qc, qubits):
    qubits=list(qubits)
    n=len(qubits)
    for i in range(n//2):
        qc.swap(qubits[i],qubits[n-1-i])
    for j in reversed(range(n)):
        for k in reversed(range(j+1,n)):
            qc.cp(-np.pi/(2**(k-j)), qubits[k], qubits[j])
        qc.h(qubits[j])

# Part IV — Pedagogical order finding for N=15, a=2
For $N=15$ and $a=2$, multiplication by powers of 2 modulo 15 can be implemented very compactly on four work qubits:

- multiply by $2$ modulo 15: cyclic left rotation of the 4-bit representation;
- multiply by $4$: rotate twice;
- multiply by $8$: rotate three times.

For phase estimation we only need controlled versions of these permutations.

In [ ]:
def controlled_mul_2_power_mod15(qc, control, work, power):
    # Multiply by 2^power mod 15 on the orbit relevant to the teaching example.
    # A cyclic left rotation of four bits is multiplication by 2 mod 15 for nonzero residues.
    rotations=power%4
    for _ in range(rotations):
        # left rotation q0<-q3, q1<-q0, q2<-q1, q3<-q2 via 3 controlled swaps
        qc.cswap(control,work[3],work[2])
        qc.cswap(control,work[2],work[1])
        qc.cswap(control,work[1],work[0])

## 5. Build the order-finding circuit
The work register starts in $|1\rangle$. Each counting qubit controls multiplication by $a^{2^j}$.

In [ ]:
def shor15_order_finding(n_count=4):
    count=QuantumRegister(n_count,'count')
    work=QuantumRegister(4,'work')
    c=ClassicalRegister(n_count,'c')
    qc=QuantumCircuit(count,work,c)
    qc.x(work[0])  # |1>
    qc.h(count)
    for j in range(n_count):
        controlled_mul_2_power_mod15(qc,count[j],list(work),(2**j)%4)
    inverse_qft(qc,list(count))
    qc.measure(count,c)
    return qc

qc=shor15_order_finding(4)
print(qc.draw(fold=120))

## 6. Execute and inspect peaks

In [ ]:
counts=run_counts(qc,4096)
print(counts)
plot_counts(counts,'Order finding for N=15, a=2')

## 7. Convert measurements into phase fractions

In [ ]:
Q=16
for bitstr,cnt in sorted(counts.items(), key=lambda kv:-kv[1])[:8]:
    m=int(bitstr,2)
    phase=m/Q
    frac=Fraction(phase).limit_denominator(15)
    print(bitstr, 'm=',m, 'm/Q=',phase, 'candidate=',frac, 'count=',cnt)

## 8. Candidate-order validation
A continued fraction suggests a denominator, but the candidate must be checked classically by verifying $a^r\equiv1\pmod N$.

In [ ]:
def candidate_orders_from_counts(counts,Q,a,N):
    out=[]
    for bitstr,cnt in sorted(counts.items(), key=lambda kv:-kv[1]):
        m=int(bitstr,2)
        if m==0: continue
        frac=Fraction(m,Q).limit_denominator(N)
        r=frac.denominator
        valid=pow(a,r,N)==1
        out.append((bitstr,m,frac,r,valid,cnt))
    return out

candidate_orders_from_counts(counts,16,2,15)[:10]

# Part V — Classical factor extraction

In [ ]:
def factors_from_order(a,r,N):
    if r%2: return None
    x=pow(a,r//2,N)
    if x in (1,N-1): return None
    return gcd(x-1,N), gcd(x+1,N)

print(factors_from_order(2,4,15))

# Part VI — Why QFT is central but not the whole algorithm
Shor is often described as “an application of QFT,” but that is incomplete. The costly and problem-specific part is **coherent modular exponentiation**. QFT/inverse-QFT extracts the phase/period information created by that arithmetic.

Main components:
1. random classical choice of $a$;
2. GCD check;
3. reversible modular arithmetic;
4. phase estimation / periodic interference;
5. inverse QFT;
6. continued fractions;
7. GCD factor extraction.

# Part VII — Scaling
For an $L$-bit integer $N$, Shor uses polynomial resources in $L$ in the ideal fault-tolerant model, while the best-known general classical factoring algorithms are super-polynomial/sub-exponential rather than polynomial. Practical cryptographically relevant factoring requires large fault-tolerant quantum computers because modular arithmetic dominates the logical-gate count.

# Part VIII — Failure modes are normal
A single run may fail because:
- chosen $a$ is not coprime to $N$ (this can actually immediately reveal a factor);
- the recovered order is odd;
- $a^{r/2}\equiv-1\pmod N$;
- the sampled phase has $k$ not coprime to $r$;
- finite precision yields an ambiguous denominator.

Shor is therefore a **probabilistic algorithm repeated until useful classical post-processing succeeds**.

# Part IX — Exercises
1. Find the order of 7 modulo 15 by hand.
2. Explain why $a^{r/2}\neq-1\pmod N$ is required.
3. For $Q=32$, $r=4$, predict the ideal Fourier peaks.
4. If a measurement gives $m/Q=3/8$, what denominators might continued fractions suggest?
5. Identify which parts of Shor are classical and which are quantum.

# Part X — Solutions
For $a=7$ mod 15: $7^2=49\equiv4$, $7^4\equiv1$, so the order is 4. For $Q=32$ and $r=4$, ideal peaks are at $0,8,16,24$. If $a^{r/2}\equiv-1$, then $a^{r/2}+1$ is divisible by $N$ and the GCD step gives only a trivial factor.

# Compact reference
**Classical reduction**
$$\text{factor }N\longrightarrow\text{find order }r\text{ of }a\pmod N.$$

**Quantum core**
$$\text{modular exponentiation}\rightarrow\text{phase/period encoding}\rightarrow QFT^{-1}\rightarrow m/Q\approx k/r.$$

**Classical recovery**
$$m/Q\xrightarrow{\text{continued fractions}}r\xrightarrow{\gcd(a^{r/2}\pm1,N)}\text{factors}.